
## 02_autoloader_silver

The KEY notebook of Project 2.
Replaces manual Bronze reading with Auto Loader.

Auto Loader core concepts:
  format("cloudFiles")     → tells Spark to use Auto Loader
  cloudFiles.format        → the format of incoming files (json)
  schemaLocation           → where to save inferred schema
  checkpointLocation       → where to track processed files
  trigger(availableNow=True) → process all pending files then stop

Why checkpointLocation matters:
  Run 1: Bronze has 4 files → Auto Loader processes all 4
  Run 2: Bronze has 6 files → Auto Loader processes only 2 (new ones)
  Run 3: Bronze has 6 files → Auto Loader processes 0 (nothing new)

Without checkpoint:
  Every run reprocesses everything → duplicates in Silver

In [0]:
%python

# Define config variables inline
BASE_PATH = "/Volumes/workspace/default/github_streaming"
BRONZE_PATH = f"{BASE_PATH}/bronze/events"
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/bronze_to_silver"
SCHEMA_PATH = f"{BASE_PATH}/schemas/bronze_schema"
SILVER_TABLE = "default.silver_github_events"

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, TimestampType
)
from delta.tables import DeltaTable
from datetime import datetime

print("Libraries loaded ✓")
print(f"Bronze path     : {BRONZE_PATH}")
print(f"Silver table    : {SILVER_TABLE}")
print(f"Checkpoint path : {CHECKPOINT_PATH}")
print(f"Schema path     : {SCHEMA_PATH}")

In [0]:
%python

# ── Define Bronze schema explicitly ──────────────────────────
# STUDY NOTE:
#   Auto Loader CAN infer schema automatically.
#   But for production, explicit schema is better:
#   PRO explicit: faster (no inference scan), predictable
#   PRO explicit: fails loudly if source changes unexpectedly
#   PRO inferred: zero maintenance, handles new fields
#   CON inferred: first run slower, may infer wrong types
#
#   We define only the fields we NEED — not all nested fields.
#   This is called "schema projection" — take only what's useful.
#
#   For nested structs we use dot notation in Silver transforms:
#   actor.login → F.col("actor.login")

from pyspark.sql.types import StructType, StructField, StringType, LongType

actor_schema = StructType([
    StructField("id",            LongType(),   True),
    StructField("login",         StringType(), True),
    StructField("display_login", StringType(), True),
    StructField("url",           StringType(), True),
])

repo_schema = StructType([
    StructField("id",   LongType(),   True),
    StructField("name", StringType(), True),
    StructField("url",  StringType(), True),
])

bronze_schema = StructType([
    StructField("id",         StringType(),  True),
    StructField("type",       StringType(),  True),
    StructField("actor",      actor_schema,  True),
    StructField("repo",       repo_schema,   True),
    StructField("created_at", StringType(),  True),
    StructField("public",     StringType(),  True),
])

print("Schema defined ✓")
print("Fields:", [f.name for f in bronze_schema.fields])

In [0]:
%python

# ── Auto Loader readStream ────────────────────────────────────
# STUDY NOTE:
#   This is the core of Auto Loader.
#   spark.readStream vs spark.read:
#
#   spark.read       → static, reads all files right now
#   spark.readStream → streaming, monitors path continuously
#
#   format("cloudFiles") activates Auto Loader.
#   Without it, readStream would need Kafka/socket source.
#
#   cloudFiles.schemaLocation:
#   → Auto Loader saves the inferred/defined schema here
#   → On subsequent runs it reuses saved schema
#   → Prevents re-scanning files just for schema
#
#   multiLine=false means NDJSON (one JSON per line)
#   This is the correct setting for GitHub Archive files

df_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("multiLine", "false")
        .schema(bronze_schema)
        .load(BRONZE_PATH)
)

print("Stream reader defined ✓")
print("Stream is NOT running yet — just defined the reader")
print("Schema of incoming stream:")
df_stream.printSchema()

In [0]:
%python

# ── Flatten nested JSON for Silver ───────────────────────────
# STUDY NOTE:
#   Raw Bronze has nested structs (actor.login, repo.name).
#   Silver flattens these to top-level columns.
#   Why flatten?
#   PRO: SQL queries don't need dot notation
#   PRO: Power BI / downstream tools work better with flat schema
#   PRO: Easier to apply filters and joins
#   CON: Loses the nested structure (fine for analytics)
#
#   F.col("actor.login") accesses nested field in PySpark
#   .alias("actor_login") gives it a flat name
#
#   _metadata is a special Auto Loader column that tells you
#   WHICH FILE each row came from — very useful for debugging
#   and tracking data lineage.

df_silver_stream = (
    df_stream
    .select(
        F.col("id")                     .alias("event_id"),
        F.col("type")                   .alias("event_type"),
        F.col("actor.id")               .alias("actor_id"),
        F.col("actor.login")            .alias("actor_login"),
        F.col("repo.id")                .alias("repo_id"),
        F.col("repo.name")              .alias("repo_name"),
        F.to_timestamp("created_at")    .alias("event_timestamp"),
        F.to_date("created_at")         .alias("event_date"),
        F.col("public")                 .alias("is_public"),
        # STUDY NOTE: _metadata.file_name tells you which
        # Bronze file this row came from — this is Auto Loader
        # data lineage tracking built-in
        F.col("_metadata.file_name")    .alias("source_file"),
        F.current_timestamp()           .alias("ingestion_timestamp"),
    )
)

print("Flattening transforms defined ✓")
print("Silver stream schema:")
df_silver_stream.printSchema()

In [0]:
%python

# ── Write stream to Silver Delta table ───────────────────────
# STUDY NOTE:
#   writeStream vs write:
#   write       → one-time batch write
#   writeStream → continuous or triggered write
#
#   trigger(availableNow=True):
#   → "Process all files currently in Bronze, then STOP"
#   → This is "incremental batch" mode
#   → Perfect for scheduled pipelines (not always-on streaming)
#   → Alternative: trigger(processingTime="10 seconds")
#     would run continuously every 10 seconds
#
#   outputMode("append"):
#   → New rows only — never update or delete
#   → Correct for event data (events are immutable)
#   → Silver events never change once written
#
#   checkpointLocation:
#   → THE MOST IMPORTANT OPTION
#   → Without this, every run reprocesses all files
#   → With this, only NEW files are processed each run
#   → Databricks manages the checkpoint automatically

print(f"Starting Auto Loader stream...")
print(f"Writing to   : {SILVER_TABLE}")
print(f"Checkpoint   : {CHECKPOINT_PATH}")
print(f"Started at   : {datetime.now()}")

query = (
    df_silver_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .trigger(availableNow=True)
        .toTable(SILVER_TABLE)
)

# Wait for stream to finish
# STUDY NOTE:
#   awaitTermination() blocks until the stream stops.
#   With availableNow=True the stream stops automatically
#   after processing all pending files.
#   Without awaitTermination(), the cell would finish
#   immediately while stream runs in background.
query.awaitTermination()

print(f"\n✓ Stream completed at: {datetime.now()}")

In [0]:
%python

# ── Verify Silver table ───────────────────────────────────────
df_silver = spark.table(SILVER_TABLE)

print("=" * 50)
print("Silver Layer Summary")
print("=" * 50)
print(f"Total rows  : {df_silver.count():,}")
print(f"Columns     : {len(df_silver.columns)}")
print("=" * 50)

# Event type breakdown
print("\nEvent types in Silver:")
display(
    df_silver
    .groupBy("event_type")
    .count()
    .orderBy("count", ascending=False)
)

In [0]:
%python

# ── Prove checkpoint is tracking files ───────────────────────
# STUDY NOTE:
#   This is the most educational cell in Project 2.
#   We prove that Auto Loader tracked the files it processed.
#
#   The checkpoint folder contains:
#   commits/    → transaction log of what was committed
#   offsets/    → which files were seen at each batch
#   sources/    → the actual file list per batch
#
#   After this cell, run 02_autoloader_silver AGAIN.
#   You should see 0 new rows added — because all 4 files
#   are already in the checkpoint.
#   Then land a NEW file using 01_simulate_landing with
#   a different hour, run again — only that new file processes.

print("Checkpoint contents:")
print("-" * 50)

try:
    checkpoint_contents = dbutils.fs.ls(CHECKPOINT_PATH)
    for item in checkpoint_contents:
        print(f"  {item.name}")
        try:
            sub_items = dbutils.fs.ls(item.path)
            for sub in sub_items:
                print(f"    └── {sub.name} ({sub.size} bytes)")
        except:
            pass
except Exception as e:
    print(f"Checkpoint path: {e}")

print("\nSource file tracking in Silver:")
display(
    df_silver
    .groupBy("source_file", "event_date")
    .count()
    .orderBy("event_date", "source_file")
)